In [2]:
# %env REGIONS=HIM
%env FORCE_SPLIT=0

env: FORCE_SPLIT=0


In [3]:
# Cell 1: Setup
import os
import polars as pl
import spatial_split as ss
from config import regions as config_regions

# Regions to process: defaults to the regions in config.py, or set via the REGIONS environment variable (comma separated)
REGIONS = os.environ.get('REGIONS', ','.join(config_regions)).split(',')
# REGIONS = ['DES']
STRESS_YEARS = {'HIM': [2003, 2001]}   # per region; regions not listed get the weakest unused usable year automatically
N_SPREAD = 8                           # evenly spread search years
plans = {}

In [4]:
# Cell 2: Usable years + search years per region (saved to data/study_years/{REGION}.json).
# Edit data/study_years/overrides.json to exclude years or fix the search years by hand, then rerun this cell.
for REGION in REGIONS:
    print(f"\n{'='*60}\n{REGION}\n{'='*60}")
    plans[REGION] = ss.plan_years(REGION, N_SPREAD, STRESS_YEARS.get(REGION))
    print(plans[REGION]['table'].to_pandas().to_string(index=False))


HIM
[HIM] usable 30 years (1993-2025); search 9: [1993, 2000, 2003, 2004, 2008, 2013, 2017, 2021, 2025]; stress 2003
 year  rows  polys  n_cls  min_cls  usable
 1987     0      0      0        0   False
 1988  1990     21      6        1   False
 1989  4344     52      9        1   False
 1990  4654     66      8        1   False
 1991 13618    169     10        3   False
 1992 31114    285     10        4   False
 1993 42967    385     11        6    True
 1994 40271    365     11        6    True
 1995  6746     87      9        1   False
 1996 35110    341     11        1   False
 1997 14301    182     11        1   False
 1998 42324    382     11        6    True
 1999 36980    374     11        6    True
 2000 43589    403     11        6    True
 2001 37030    378     11        6    True
 2002 41083    378     11        6    True
 2003 35184    382     11        6    True
 2004 41107    386     11        6    True
 2005 41165    397     11        6    True
 2006 42702    401    

In [5]:
# Cell 3: What was chosen
for REGION, plan in plans.items():
    print(f"[{REGION}] usable {len(plan['usable'])}: {plan['usable']}")
    print(f"[{REGION}] search {len(plan['search'])}: {plan['search']} | stress year: {plan['stress']}")

[HIM] usable 30: [1993, 1994, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
[HIM] search 9: [1993, 2000, 2003, 2004, 2008, 2013, 2017, 2021, 2025] | stress year: 2003
[IGP] usable 30: [1992, 1993, 1994, 1996, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
[IGP] search 9: [1992, 1996, 2000, 2004, 2008, 2013, 2017, 2021, 2025] | stress year: 1996
[DES] usable 34: [1991, 1992, 1993, 1994, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
[DES] search 9: [1991, 1997, 2001, 2002, 2006, 2011, 2016, 2020, 2025] | stress year: 2002
[PEN] usable 21: [2000, 2003, 2004, 2006, 2007, 2008, 2009, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2

In [6]:
# Cell 4: Build the polygon-level split (70:20:10) on the usable years. ~minutes per region.
# This run needs a NEW split (new fractions, polygon pinning for classes with < ss.MIN_POLYS polygons), so it is rebuilt by default.
# Set FORCE_SPLIT=0 to reuse an existing lookup instead (only once the new split is built).
FORCE = os.environ.get("FORCE_SPLIT", "1") == "1"
for REGION in REGIONS:
    print(f"\n--- {REGION} ---")
    ss.build_block_splits(REGION, force=FORCE)



--- HIM ---

--- IGP ---

--- DES ---

--- PEN ---

--- NEI ---

--- WGR ---

--- ICR ---


In [7]:
# Cell 5: Check the split per year. Look for years where val/test miss a class, or are tiny.
for REGION in REGIONS:
    s = ss.split_summary(REGION)
    print(f"\n--- {REGION} ---")
    print(s.to_pandas().to_string(index=False))
    bad = s.filter((pl.col('val_missing') != '') | (pl.col('test_missing') != ''))
    print(f"{REGION}: {bad.height} of {s.height} years have a class missing from val or test")


--- HIM ---
 year  train_px  train_polys  val_px  val_polys val_missing  test_px  test_polys test_missing
 1993     30717          187    8236        114                 4014          84             
 1994     28602          174    8065        109                 3604          82             
 1998     30314          191    8049        112                 3961          79             
 1999     25758          183    7397        111                 3825          80             
 2000     31237          194    8268        120                 4084          89             
 2001     25408          186    7728        111                 3894          81             
 2002     29489          182    7756        114                 3838          82             
 2003     24744          189    6946        114                 3494          79             
 2004     29417          190    7740        113                 3950          83             
 2005     29179          195    8056        117

In [8]:
# Cell 6: Polygon sanity check (same polygon -> one class; polygons stay whole in one split)
for REGION in REGIONS:
    print(f"\n--- {REGION} ---")
    ss.polygon_check(REGION, years=ss.study_years(REGION)[:3])


--- HIM ---
year=1993: 45,029 rows, 0 without area_m2, 420 polygons, pixels/polygon median 40, p90 225
year=1994: 44,886 rows, 0 without area_m2, 422 polygons, pixels/polygon median 38, p90 224
year=1998: 44,966 rows, 0 without area_m2, 419 polygons, pixels/polygon median 40, p90 224

year=1993 vs year=1998: 0 shared system:index values (0.0% of year=1993); same lat/lon on 0; same area_m2 on 0
If shared == 0 the two years sample DIFFERENT pixels; if shared is large and lat/lon agree, the same pixels are reused.
polygon extent km (median, p90, p99, max): 0.48, 1.89, 5.52, 7.80;  keys with >1 class: 0

--- IGP ---
year=1992: 82,587 rows, 0 without area_m2, 556 polygons, pixels/polygon median 55, p90 386
year=1993: 82,587 rows, 0 without area_m2, 556 polygons, pixels/polygon median 55, p90 386
year=1994: 82,587 rows, 0 without area_m2, 556 polygons, pixels/polygon median 55, p90 386

year=1992 vs year=1994: 0 shared system:index values (0.0% of year=1992); same lat/lon on 0; same area_m2

In [9]:
# Cell 7: What the later steps will use
for REGION in REGIONS:
    print(f"{REGION}: usable pool (split, 04 ranking, 06 pruning) {len(ss.study_years(REGION))} years | "
          f"search years (07 Optuna) {len(ss.search_years(REGION))}: {ss.search_years(REGION)}")

HIM: usable pool (split, 04 ranking, 06 pruning) 30 years | search years (07 Optuna) 9: [1993, 2000, 2003, 2004, 2008, 2013, 2017, 2021, 2025]
IGP: usable pool (split, 04 ranking, 06 pruning) 30 years | search years (07 Optuna) 9: [1992, 1996, 2000, 2004, 2008, 2013, 2017, 2021, 2025]
DES: usable pool (split, 04 ranking, 06 pruning) 34 years | search years (07 Optuna) 9: [1991, 1997, 2001, 2002, 2006, 2011, 2016, 2020, 2025]
PEN: usable pool (split, 04 ranking, 06 pruning) 21 years | search years (07 Optuna) 9: [2000, 2003, 2006, 2009, 2014, 2016, 2019, 2022, 2025]
NEI: usable pool (split, 04 ranking, 06 pruning) 13 years | search years (07 Optuna) 9: [2013, 2014, 2015, 2016, 2018, 2020, 2022, 2023, 2025]
WGR: usable pool (split, 04 ranking, 06 pruning) 19 years | search years (07 Optuna) 9: [2000, 2006, 2007, 2012, 2015, 2017, 2020, 2022, 2025]
ICR: usable pool (split, 04 ranking, 06 pruning) 24 years | search years (07 Optuna) 9: [2001, 2004, 2008, 2011, 2012, 2015, 2018, 2022, 2025]

In [10]:
# Cell 8: Data gaps per region = classes with fewer than ss.MIN_POLYS polygons. They are scored (ss.SCORE_FEW_POLY=True) but flagged
# low-confidence, and go into the per-region action items for the groups (collect more / larger reference polygons).
import pandas as pd
for REGION in REGIONS:
    scored, gaps = ss.scored_classes(REGION)
    print(f"{REGION}: {len(scored)} classes scored; low-confidence (< {ss.MIN_POLYS} polygons): {gaps if gaps else 'none'}")


HIM: 11 classes scored; low-confidence (< 10 polygons): [11]
IGP: 10 classes scored; low-confidence (< 10 polygons): none
DES: 11 classes scored; low-confidence (< 10 polygons): [61]
PEN: 11 classes scored; low-confidence (< 10 polygons): none
NEI: 11 classes scored; low-confidence (< 10 polygons): none
WGR: 11 classes scored; low-confidence (< 10 polygons): none
ICR: 11 classes scored; low-confidence (< 10 polygons): none
